# NotebookLM Kit — Notebook Inventory

Read-only view of a notebook's contents.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Sources** — list all sources (indexed)
4. **Artifacts** — list all artifacts with their type, creation time, and the indices of the sources used to generate each one (referencing the table above)

In [ ]:
import importlib, config, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts, pipeline.config
importlib.reload(config)
importlib.reload(pipeline.config)
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from config import NOTEBOOK_ID
from pipeline import load_credentials, check_tsx, login

PROFILE = "kd"

login(profile=PROFILE)
creds = load_credentials(mode="patchright", profile=PROFILE)
check_tsx()

# Find your notebook ID in the NotebookLM URL: `https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**
# NOTEBOOK_ID = "00000000-0000-0000-0000-000000000000"  # ← paste your notebook ID here

Credentials ready [kd] — token: 42 chars, cookies: 2540 chars
tsx: tsx v4.22.3
node v20.17.0


## Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources.  
The index column is what the **Artifacts** table below uses to reference each source.

In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)


Notebook : Foundations and Responsibilities of the Chartered Engineer
+---+---------------------------------------------------+------------+--------------------------------------+
| # | Title                                             | Status     | Source ID                            |
+---+---------------------------------------------------+------------+--------------------------------------+
| 0 | 00 Philosophy of Ethics.md                        | READY      | 557aa636-a9b8-48bb-93ff-d1872866a9e0 |
| 1 | 01 Role of an Engineer.md                         | READY      | 8ba915da-9374-4ebf-8bbc-512a2216c850 |
| 2 | 02 Engineering Ethics 1.md                        | READY      | 2ec089ed-154b-46fb-a0ad-e059648b519c |
| 3 | 03 Engineering Ethics 2.md                        | READY      | 02666893-a6e5-4706-8e83-0d354ce12858 |
| 4 | 04 Professional and Regulatory Bodies.md          | READY      | e9817c27-e84b-4f7b-b994-45e24fa273fd |
| 5 | 05 Case Study 1.md                         

## Artifacts

`list_artifacts(notebook_id, sources, creds)` — every artifact in the notebook with:

- **Title** and **Type** (FLASHCARDS, QUIZ, VIDEO, …)
- **Created** — local time pulled from the artifact's own `createdAt`
- **Sources** — comma-separated **indices** into the sources table above (e.g. `0,2` means rows 0 and 2)

In [ ]:
from pipeline import list_artifacts

artifacts = list_artifacts(NOTEBOOK_ID, sources, creds)


Artifacts in notebook 60007ecf-03f4-4fde-af9b-f1e2c422aa49 (30 total)
+----+----------------------------------------------------------------+-------------+---------------------+---------+--------------------------------------+
| #  | Title                                                          | Type        | Created             | Sources | Artifact ID                          |
+----+----------------------------------------------------------------+-------------+---------------------+---------+--------------------------------------+
|  0 | Defensible Decisions                                           | VIDEO       | 2026-06-12 07:54:02 | 8       | b174a921-3df4-4105-9b49-400d306b6a09 |
|  1 | 07 CSE Perspective [260612 073954]                             | VIDEO       | 2026-06-12 07:39:54 | 7       | 3f95746c-8789-41c8-96e3-2c65c877dacb |
|  2 | 03 Engineering Ethics 2 [260612 073946]                        | VIDEO       | 2026-06-12 07:39:46 | 3       | cba0ad44-14d1-43cf-8509-05

## Rename Single-Source Artifacts

`rename_single_source_artifacts(artifacts, sources, creds, *, indices=None, dry_run=False)`

For every artifact whose `sourceIds` contains exactly **one** source, rename it to:

```
<source title> YYMMDD HHMM
```

where the timestamp comes from the artifact's own `createdAt` (local time).

- `indices` — optional list of row numbers from the Artifacts table above to limit the operation; `None` means all single-source artifacts.
- `dry_run=True` — preview the planned renames without calling the API.

Multi-source artifacts (e.g. audio overviews built from all sources) are left untouched.

In [ ]:
from pipeline import rename_single_source_artifacts

# Preview only — no changes made
rename_single_source_artifacts(artifacts, sources, creds, dry_run=False)

# Limit to specific rows from the Artifacts table:
# rename_single_source_artifacts(artifacts, sources, creds, indices=[0, 1, 3], dry_run=True)


Renaming 1 single-source artifact(s)
+----+----------------------+----------------------------------------+----------+
| #  | Old title            | New title                              | Status   |
+----+----------------------+----------------------------------------+----------+
|  0 | Defensible Decisions | 08 Professional Ethics [260612 075402] | ok       |
+----+----------------------+----------------------------------------+----------+


[{'index': 0,
  'artifactId': 'b174a921-3df4-4105-9b49-400d306b6a09',
  'oldTitle': 'Defensible Decisions',
  'newTitle': '08 Professional Ethics [260612 075402]',
  'status': 'ok',
  'error': None}]

## Download Artifacts by Type

`download_artifacts_by_type(artifacts, artifact_type, notebook_id, creds, *, sources=None, indices=None, output_dir=None)`

Downloads every artifact of the given type from the artifacts list (or a subset via `indices`).

- **Single-source** artifacts are named after their **source** (pass `sources=sources`).
- **Multi-source** artifacts (e.g. audio overview from all sources) use the artifact title.
- Already-downloaded files are skipped automatically.

Valid types: `FLASHCARDS`, `QUIZ`, `AUDIO`, `INFOGRAPHIC`, `VIDEO`, `SLIDE_DECK`.

In [ ]:
from pipeline import download_artifacts_by_type

download_artifacts_by_type(artifacts, "INFOGRAPHIC", NOTEBOOK_ID, creds)

# sources= is optional — omit it and sources are fetched automatically.
# Limit to specific rows from the Artifacts table:
# download_artifacts_by_type(artifacts, "VIDEO", NOTEBOOK_ID, creds, indices=[0, 2, 5])
# download_artifacts_by_type(artifacts[2:6], "AUDIO", NOTEBOOK_ID, creds)


Downloaded 10 / 10 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\infographic\s6 Human Computer Interaction
+---+---------------------------------+----------------------------------------------+------------+
| # | Source                          | File                                         | Status     |
+---+---------------------------------+----------------------------------------------+------------+
| 0 | 11 User Support.md              | 11 User Support [20260608 164240].png        | downloaded |
| 1 | 10 Universal Design.md          | 10 Universal Design [20260608 164237].png    | downloaded |
| 2 | 08 Implementation Support.md    | 08 Implementation Support [20260608 16423... | downloaded |
| 3 | 07 Design Rules.md              | 07 Design Rules [20260608 164230].png        | downloaded |
| 4 | 06 Heuristics.md                | 06 Heuristics [20260608 164228].png          | downloaded |
| 5 | 09 Evalutaion Techniques.md     | 09 Evalutaion Techniques [20260608 164235..

[{'sourceTitle': '11 User Support.md',
  'notebookTitle': 's6 Human Computer Interaction',
  'createdAt': '2026-06-08T11:12:40.119Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\infographic\\s6 Human Computer Interaction\\11 User Support [20260608 164240].png',
  'status': 'downloaded'},
 {'sourceTitle': '10 Universal Design.md',
  'notebookTitle': 's6 Human Computer Interaction',
  'createdAt': '2026-06-08T11:12:37.765Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\infographic\\s6 Human Computer Interaction\\10 Universal Design [20260608 164237].png',
  'status': 'downloaded'},
 {'sourceTitle': '08 Implementation Support.md',
  'notebookTitle': 's6 Human Computer Interaction',
  'createdAt': '2026-06-08T11:12:32.828Z',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\infographic\\s6 Human Computer Interaction\\08 Implementation Support [20260608 164232].png',
  'status': 'downloaded'},
 {'sourceTitle': '07 Design Rules.md',
  'notebookTitle': 's